# Phase 3 — Joint CL + IT rhetorical-role classification


In [1]:
%pip install -q huggingface_hub tokenizers scikit-learn tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, random, gc, tempfile, zipfile
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'encoder_decoder_mixed'
MODEL_PATH = 'checkpoints/encoder_decoder_mixed/model.pt'
TOKENIZER_PATH = 'tokenizer_encoder_decoder/tokenizer.json'
MODEL_CONFIG = dict(
    dim=640,
    encoder_layers=6,
    decoder_layers=6,
    n_heads=16,
    n_kv_heads=4,
    ff_dim=1728,
    max_encoder_len=512,
    max_decoder_len=256,
)
MAX_SEQ_LEN = 512
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 20
PATIENCE = 5
MIN_DELTA = 0.001
ADAPTER_CONFIG = dict(r=16, alpha=32, dropout=0.05, do_pi=True)
HEAD_DROPOUT = 0.1
RESUME_TAGS = {'JOINT': None}  # Set to the joint latest tag to resume
UPLOAD_CHECKPOINTS = True
WORK_DIR = Path(f"/kaggle/working/rr_{PHASE1_TAG}_lorma_joint") if Path('/kaggle/working').exists() else Path.cwd() / 'rr_decoder126m_lorma_joint'
WORK_DIR.mkdir(parents=True, exist_ok=True)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)

Device: cuda


In [3]:
local_tokenizer = WORK_DIR / 'tokenizer.json'
fs.get_file(f'{BUCKET_ROOT}/{TOKENIZER_PATH}', str(local_tokenizer))
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
PAD_ID = tokenizer.token_to_id('<|pad|>')
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
assert PAD_ID is not None and EMPTY_ID is not None

EXPECTED_LABELS = ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent',
                   'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled',
                   'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']

def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f:
        return json.load(f)

vocab = download_json('Phase_3_RR/Train/label_vocab.json')
if isinstance(vocab, list) and all(isinstance(x, str) for x in vocab):
    LABELS = vocab
elif isinstance(vocab, dict) and set(vocab) == set(EXPECTED_LABELS):
    LABELS = sorted(vocab, key=lambda x: int(vocab[x])) if all(isinstance(v, int) for v in vocab.values()) else list(vocab)
elif isinstance(vocab, dict) and set(vocab.values()) == set(EXPECTED_LABELS):
    LABELS = [vocab[k] for k in sorted(vocab, key=lambda x: int(x))]
else:
    raise ValueError(f'Unexpected label_vocab format: {vocab!r}')
assert len(LABELS) == 13 and set(LABELS) == set(EXPECTED_LABELS)
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = dict(enumerate(LABELS))

def normalize_documents(raw, labelled):
    if isinstance(raw, list):
        docs = raw
    elif isinstance(raw, dict) and 'id' in raw and 'text' in raw:
        docs = [raw]
    elif isinstance(raw, dict):
        docs = []
        for doc_id, value in raw.items():
            if isinstance(value, list) and not labelled:
                doc = {'id': doc_id, 'text': value}
            elif isinstance(value, dict):
                doc = dict(value)
                doc.setdefault('id', doc_id)
                assert doc['id'] == doc_id, 'Document ID mismatch'
            else:
                raise ValueError(f'Unexpected value for document {doc_id!r}: {type(value).__name__}')
            docs.append(doc)
    else:
        raise ValueError('Unsupported document container')
    result, seen = [], set()
    for doc in docs:
        doc = dict(doc)
        assert isinstance(doc['id'], str), 'Document IDs must be strings'
        assert doc['id'] not in seen, f"Duplicate ID: {doc['id']}"
        seen.add(doc['id'])
        assert isinstance(doc['text'], list) and all(isinstance(s, str) for s in doc['text'])
        if labelled:
            assert len(doc['labels']) == len(doc['text'])
            assert all(label in label2id for label in doc['labels'])
        if not labelled:
            doc.pop("labels", None)
        result.append(doc)
    return result

data = {}
for domain in ['CL', 'IT']:
    data[domain] = {}
    for split in ['train', 'dev', 'test']:
        folder = 'Test' if split == 'test' else 'Train'
        data[domain][split] = normalize_documents(
            download_json(f'Phase_3_RR/{folder}/{domain}_{split}.json'), split != 'test')
    assert not ({d['id'] for d in data[domain]['train']} & {d['id'] for d in data[domain]['dev']})
    for split, docs in data[domain].items():
        print(domain, split, 'documents:', len(docs), 'sentences:', sum(len(d['text']) for d in docs))
    print('Training label counts:', dict(Counter(l for d in data[domain]['train'] for l in d['labels'])))
assert not ({d['id'] for d in data['CL']['test']} & {d['id'] for d in data['IT']['test']}), 'Test IDs collide across domains'
print('Label mapping:', label2id)

(…)tokenizer_encoder_decoder/tokenizer.json:   0%|          | 0.00/2.39M [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

CL train documents: 40 sentences: 10593
CL dev documents: 5 sentences: 1396
CL test documents: 5 sentences: 1339
Training label counts: {'Fact': 2979, 'ArgumentPetitioner': 976, 'ArgumentRespondent': 996, 'RatioOfTheDecision': 2856, 'RulingByPresentCourt': 113, 'RulingByLowerCourt': 438, 'None': 53, 'PrecedentNotReliedUpon': 209, 'PrecedentReliedUpon': 1305, 'Statute': 367, 'Issue': 17, 'Dissent': 284}


IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

IT train documents: 40 sentences: 6252
IT dev documents: 5 sentences: 746
IT test documents: 5 sentences: 858
Training label counts: {'Fact': 1119, 'RulingByLowerCourt': 436, 'ArgumentPetitioner': 637, 'Issue': 90, 'Statute': 732, 'RatioOfTheDecision': 1445, 'RulingByPresentCourt': 194, 'PrecedentReliedUpon': 640, 'ArgumentRespondent': 352, 'PrecedentNotReliedUpon': 360, 'None': 244, 'PrecedentOverruled': 3}
Label mapping: {'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}


In [4]:
tokenizer.get_vocab_size()

33178

In [5]:
class RRSentenceDataset(Dataset):
    def __init__(self, documents, domain):
        self.rows = []
        truncated = empty = 0
        for doc in documents:
            for index, sentence in enumerate(doc['text']):
                ids = tokenizer.encode(sentence, add_special_tokens=False).ids
                truncated += len(ids) > MAX_SEQ_LEN
                empty += len(ids) == 0
                ids = ids[:MAX_SEQ_LEN] or [EMPTY_ID]
                self.rows.append(dict(input_ids=ids, doc_id=doc['id'], sentence_index=index,
                                      domain=domain, label=label2id[doc['labels'][index]] if 'labels' in doc else -100))
        print(domain, 'examples:', len(self.rows), 'truncated:', truncated, 'empty:', empty)
    def __len__(self): return len(self.rows)
    def __getitem__(self, index): return self.rows[index]

def collate_rr(rows):
    lengths = torch.tensor([len(r['input_ids']) for r in rows], dtype=torch.long)
    ids = torch.full((len(rows), int(lengths.max())), PAD_ID, dtype=torch.long)
    for i, r in enumerate(rows):
        ids[i, :lengths[i]] = torch.tensor(r['input_ids'])
    return dict(input_ids=ids, lengths=lengths,
                labels=torch.tensor([r['label'] for r in rows]),
                doc_ids=[r['doc_id'] for r in rows], indices=[r['sentence_index'] for r in rows])

datasets = {domain: {split: RRSentenceDataset(docs, domain) for split, docs in splits.items()}
            for domain, splits in data.items()}
def make_loader(domain, split):
    return DataLoader(datasets[domain][split], batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                      collate_fn=collate_rr, num_workers=0, pin_memory=(DEVICE == 'cuda'))
from torch.utils.data import ConcatDataset
datasets['JOINT'] = {split: ConcatDataset([datasets['CL'][split], datasets['IT'][split]]) for split in ['train', 'dev']}
assert len(datasets['JOINT']['train']) == len(datasets['CL']['train']) + len(datasets['IT']['train'])


CL examples: 10593 truncated: 3 empty: 0
CL examples: 1396 truncated: 0 empty: 0
CL examples: 1339 truncated: 1 empty: 0
IT examples: 6252 truncated: 3 empty: 0
IT examples: 746 truncated: 0 empty: 0
IT examples: 858 truncated: 0 empty: 0


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DIM = 640
ENCODER_LAYERS = 6
DECODER_LAYERS = 6
N_HEADS = 16
N_KV_HEADS = 4
FF_DIM = 1728
MAX_ENCODER_LEN = 512
MAX_DECODER_LEN = 256
RMS_NORM_EPS = 1e-6
ROPE_BASE = 10000
GRAD_CLIP = 5.0

MODEL_CONFIG = dict(
    dim=DIM,
    encoder_layers=ENCODER_LAYERS,
    decoder_layers=DECODER_LAYERS,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    ff_dim=FF_DIM,
    max_encoder_len=MAX_ENCODER_LEN,
    max_decoder_len=MAX_DECODER_LEN,
)

# ============================================================
# RMSNorm
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=RMS_NORM_EPS):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        rms = (x*x).mean(
            dim=-1,
            keepdim=True
        )

        return self.weight * x * torch.rsqrt(
            rms + self.eps
        )


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=ROPE_BASE,
    ):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2,
                    dtype=torch.float32,
                ) / head_dim
            )
        )

        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32,
        )

        freqs = torch.outer(
            positions,
            inv_freq,
        )

        # [T, head_dim]
        emb = torch.cat(
            [freqs, freqs],
            dim=-1,
        )

        self.register_buffer(
            "cos",
            emb.cos()[None, None, :, :],
            persistent=False,
        )

        self.register_buffer(
            "sin",
            emb.sin()[None, None, :, :],
            persistent=False,
        )

    def forward(self, seq_len):

        return (
            self.cos[:, :, :seq_len],
            self.sin[:, :, :seq_len],
        )


def rotate_half(x):

    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]

    return torch.cat(
        [-x2, x1],
        dim=-1,
    )


def apply_rope(q, k, cos, sin):

    q = (
        q * cos
        +
        rotate_half(q) * sin
    )

    k = (
        k * cos
        +
        rotate_half(k) * sin
    )

    return q, k


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):

    def __init__(
        self,
        dim,
        hidden_dim,
    ):
        super().__init__()

        self.gate_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.up_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.down_proj = nn.Linear(
            hidden_dim,
            dim,
            bias=False,
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(
                self.gate_proj(x)
            )
            *
            self.up_proj(x)
        )


# ============================================================
# GQA + Flash Attention
# ============================================================

class Attention(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        max_seq_len,
        causal=False,
        use_rope=True,
    ):
        super().__init__()

        assert dim % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.dim = dim
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads

        self.head_dim = dim // n_heads

        self.causal = causal
        self.use_rope = use_rope

        # ----------------------------------------------------
        # Q / K / V projections
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            dim,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.o_proj = nn.Linear(
            n_heads * self.head_dim,
            dim,
            bias=False,
        )

        # RoPE only needs to be constructed for self-attention
        if use_rope:

            self.rope = RotaryEmbedding(
                self.head_dim,
                max_seq_len,
            )

    def forward(
        self,
        x,
        kv=None,
        past_kv=None,
        use_cache=False,
    ):

        B, T, C = x.shape

        # ====================================================
        # Queries
        # ====================================================

        q = self.q_proj(x)

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        )

        q = q.transpose(1, 2)

        # ====================================================
        # Keys / Values
        # ====================================================

        # Self-attention:
        #     source = x
        #
        # Cross-attention:
        #     source = encoder output

        source = x if kv is None else kv

        S = source.shape[1]

        k = self.k_proj(source)
        v = self.v_proj(source)

        k = k.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        v = v.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # ====================================================
        # RoPE
        # ====================================================

        # Only self-attention gets RoPE.
        # Cross-attention does NOT use RoPE.

        if kv is None and self.use_rope:

            current_len = k.shape[2]

            cos, sin = self.rope(
                current_len
            )

            q, k = apply_rope(
                q,
                k,
                cos,
                sin,
            )

        # ====================================================
        # Decoder KV Cache
        # ====================================================

        if (
            kv is None
            and past_kv is not None
        ):

            past_k, past_v = past_kv

            k = torch.cat(
                [past_k, k],
                dim=2,
            )

            v = torch.cat(
                [past_v, v],
                dim=2,
            )

        # ====================================================
        # GQA
        # ====================================================

        if self.n_kv_heads != self.n_heads:

            repeat_factor = (
                self.n_heads
                // self.n_kv_heads
            )

            k = k.repeat_interleave(
                repeat_factor,
                dim=1,
            )

            v = v.repeat_interleave(
                repeat_factor,
                dim=1,
            )

        # ====================================================
        # Flash / SDPA Attention
        # ====================================================

        # During cached decoding, only one new token is
        # queried, so causal masking is unnecessary.

        is_causal = (
            self.causal
            and kv is None
            and past_kv is None
        )

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=0.0,
            is_causal=is_causal,
        )

        # ====================================================
        # Output projection
        # ====================================================

        out = out.transpose(
            1,
            2,
        ).contiguous()

        out = out.view(
            B,
            T,
            C,
        )

        out = self.o_proj(out)

        # Cache only decoder self-attention
        new_kv = None

        if use_cache and kv is None:
            new_kv = (k, v)

        return out, new_kv


# ============================================================
# Encoder Block
# ============================================================

class EncoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_seq_len,
    ):
        super().__init__()

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_seq_len,
            causal=False,
            use_rope=True,
        )

        self.norm2 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(self, x):

        # Pre-norm self-attention
        x = x + self.self_attn(
            self.norm1(x)
        )[0]

        # Pre-norm FFN
        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_self_attn_len,
        max_cross_attn_len,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Causal self-attention
        # ----------------------------------------------------

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_self_attn_len,
            causal=True,
            use_rope=True,
        )

        # ----------------------------------------------------
        # Encoder-decoder cross-attention
        # ----------------------------------------------------

        self.norm2 = RMSNorm(dim)

        self.cross_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_cross_attn_len,
            causal=False,
            use_rope=False,
        )

        # ----------------------------------------------------
        # FFN
        # ----------------------------------------------------

        self.norm3 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(
        self,
        x,
        encoder_output,
        past_kv=None,
        use_cache=False,
    ):

        # ====================================================
        # 1. Decoder causal self-attention
        # ====================================================

        residual = x

        x_norm = self.norm1(x)

        self_out, new_kv = self.self_attn(
            x_norm,
            past_kv=past_kv,
            use_cache=use_cache,
        )

        x = residual + self_out

        # ====================================================
        # 2. Cross-attention
        # ====================================================

        residual = x

        x_norm = self.norm2(x)

        cross_out, _ = self.cross_attn(
            x_norm,
            kv=encoder_output,
        )

        x = residual + cross_out

        # ====================================================
        # 3. FFN
        # ====================================================

        x = x + self.ffn(
            self.norm3(x)
        )

        return x, new_kv


# ============================================================
# Encoder-Decoder Language Model
# ============================================================

class EncoderDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        dim=DIM,
        encoder_layers=ENCODER_LAYERS,
        decoder_layers=DECODER_LAYERS,
        n_heads=N_HEADS,
        n_kv_heads=N_KV_HEADS,
        ff_dim=FF_DIM,
        max_encoder_len=MAX_ENCODER_LEN,
        max_decoder_len=MAX_DECODER_LEN,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # ====================================================
        # Shared token embedding
        # ====================================================

        self.token_embedding = nn.Embedding(
            vocab_size,
            dim,
        )

        # ====================================================
        # Encoder
        # ====================================================

        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_seq_len=max_encoder_len,
            )
            for _ in range(encoder_layers)
        ])

        self.encoder_norm = RMSNorm(dim)

        # ====================================================
        # Decoder
        # ====================================================

        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_self_attn_len=256,
                max_cross_attn_len=max_encoder_len,
            )
            for _ in range(decoder_layers)
        ])

        self.decoder_norm = RMSNorm(dim)

        # ====================================================
        # Output head
        # ====================================================

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False,
        )

        # Tie output projection to token embedding
        self.lm_head.weight = (
            self.token_embedding.weight
        )

    # ========================================================
    # ENCODER
    # ========================================================

    def encode(self, encoder_ids):

        x = self.token_embedding(
            encoder_ids
        )

        for block in self.encoder_blocks:
            x = block(x)

        x = self.encoder_norm(x)

        return x

    # ========================================================
    # DECODER
    # ========================================================

    def decode(
        self,
        decoder_ids,
        encoder_output,
        past_kv=None,
        use_cache=False,
    ):

        x = self.token_embedding(
            decoder_ids
        )

        new_cache = []

        for i, block in enumerate(
            self.decoder_blocks
        ):

            layer_past = None

            if past_kv is not None:
                layer_past = past_kv[i]

            x, layer_cache = block(
                x=x,
                encoder_output=encoder_output,
                past_kv=layer_past,
                use_cache=use_cache,
            )

            if use_cache:
                new_cache.append(
                    layer_cache
                )

        x = self.decoder_norm(x)

        logits = self.lm_head(x)

        if use_cache:
            return logits, new_cache

        return logits, None

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        encoder_ids,
        decoder_ids,
        labels=None,
    ):

        # Encode 512-token context
        encoder_output = self.encode(
            encoder_ids
        )

        # Decode continuation
        logits, _ = self.decode(
            decoder_ids,
            encoder_output,
        )

        # No labels -> inference
        if labels is None:
            return logits

        # ====================================================
        # Language-model loss
        # ====================================================

        loss = F.cross_entropy(
            logits.reshape(
                -1,
                self.vocab_size,
            ),
            labels.reshape(-1),
            ignore_index=-100,
        )

        return logits, loss

    # ========================================================
    # GENERATION
    # ========================================================

    @torch.no_grad()
    def generate(
        self,
        encoder_ids,
        bos_id,
        eos_id=None,
        max_new_tokens=128,
        temperature=1.0,
        top_p=0.9,
    ):

        self.eval()

        # ----------------------------------------------------
        # Encode context ONCE
        # ----------------------------------------------------

        encoder_output = self.encode(
            encoder_ids
        )

        # ----------------------------------------------------
        # Start decoder
        # ----------------------------------------------------

        B = encoder_ids.size(0)

        decoder_ids = torch.full(
            (B, 1),
            bos_id,
            dtype=torch.long,
            device=encoder_ids.device,
        )

        past_kv = None

        # ----------------------------------------------------
        # Autoregressive generation
        # ----------------------------------------------------

        for step in range(max_new_tokens):

            # Only feed the newest token once cache exists
            if past_kv is None:
                current_ids = decoder_ids
            else:
                current_ids = decoder_ids[:, -1:]

            logits, past_kv = self.decode(
                decoder_ids=current_ids,
                encoder_output=encoder_output,
                past_kv=past_kv,
                use_cache=True,
            )

            next_logits = logits[:, -1, :]

            # Temperature
            if temperature != 1.0:
                next_logits = (
                    next_logits / temperature
                )

            # ------------------------------------------------
            # Top-p sampling
            # ------------------------------------------------

            probs = F.softmax(
                next_logits,
                dim=-1,
            )

            sorted_probs, sorted_indices = torch.sort(
                probs,
                descending=True,
                dim=-1,
            )

            cumulative_probs = torch.cumsum(
                sorted_probs,
                dim=-1,
            )

            remove = (
                cumulative_probs > top_p
            )

            # Keep at least the first token
            remove[:, 0] = False

            sorted_probs[remove] = 0.0

            sorted_probs = (
                sorted_probs
                /
                sorted_probs.sum(
                    dim=-1,
                    keepdim=True,
                )
            )

            next_token = torch.multinomial(
                sorted_probs,
                num_samples=1,
            )

            next_token = torch.gather(
                sorted_indices,
                -1,
                next_token,
            )

            decoder_ids = torch.cat(
                [
                    decoder_ids,
                    next_token,
                ],
                dim=1,
            )

            if (
                eos_id is not None
                and torch.all(
                    next_token == eos_id
                )
            ):
                break

        # Remove BOS
        return decoder_ids[:, 1:]


# ============================================================
# Parameter count
# ============================================================


In [7]:
class LoRMALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=32, dropout=0.05, do_pi=True):
        super().__init__()
        self.original_linear = original_linear
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.scaling, self.do_pi = alpha / r, do_pi
        self.dropout = nn.Dropout(dropout)
        for p in original_linear.parameters(): p.requires_grad = False
        W = original_linear.weight
        self.A = nn.Parameter(W.new_empty(r, self.out_features))
        self.B = nn.Parameter(W.new_zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def effective_weight(self):
        W = self.original_linear.weight
        delta = self.B @ self.A
        if self.do_pi:
            rows = torch.arange(self.out_features, device=W.device)[:, None]
            cols = torch.arange(self.out_features, device=W.device)[None, :]
            delta = delta.gather(1, (cols - rows) % self.out_features)
        return W + self.scaling * (delta @ W)
    def forward(self, x):
        W = self.original_linear.weight
        W_eff = self.effective_weight()
        return F.linear(x, W, self.original_linear.bias) + F.linear(self.dropout(x), W_eff - W)

class EfficientDecoderLMForRR(EncoderDecoderLM):
    def __init__(
        self,
        vocab_size,
        num_labels=13,
        head_dropout=0.1,
        decoder_start_id=None,
        **kwargs,
    ):
        super().__init__(
            vocab_size=vocab_size,
            **kwargs,
        )

        del self.lm_head

        # Uses the existing tokenizer's EOS ID unless specified.
        self.decoder_start_id = (
            EMPTY_ID if decoder_start_id is None else decoder_start_id
        )

        self.rr_dropout = nn.Dropout(head_dropout)
        self.rr_head = nn.Linear(self.dim, num_labels)

    def forward(self, input_ids, lengths):
        sentence_hidden = []

        # Your encoder has no padding mask, so trim each sentence
        # before encoding to prevent attention to padding.
        for b in range(input_ids.size(0)):
            length = int(lengths[b].item())
            encoder_ids = input_ids[b:b + 1, :length]

            encoder_output = self.encode(encoder_ids)

            decoder_ids = torch.full(
                (1, 1),
                self.decoder_start_id,
                dtype=torch.long,
                device=input_ids.device,
            )

            x = self.token_embedding(decoder_ids)

            for block in self.decoder_blocks:
                x, _ = block(
                    x,
                    encoder_output,
                    use_cache=False,
                )

            x = self.decoder_norm(x)
            sentence_hidden.append(x[:, 0, :])

        pooled = torch.cat(sentence_hidden, dim=0)

        return self.rr_head(self.rr_dropout(pooled))

phase1_path = WORK_DIR / 'latest_checkpoint.pt'
if not phase1_path.exists():
    fs.get_file(f'{BUCKET_ROOT}/{MODEL_PATH}', str(phase1_path))
def install_adapters(attention):
    for name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        original = getattr(attention, name)
        setattr(
            attention,
            name,
            LoRMALinear(original, **ADAPTER_CONFIG),
        )

def build_rr_model():
    model = EfficientDecoderLMForRR(VOCAB_SIZE, head_dropout=HEAD_DROPOUT, **MODEL_CONFIG)
    checkpoint = torch.load(phase1_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in checkpoint:
        state = checkpoint['model_state_dict']
    elif 'model' in checkpoint:
        state = checkpoint['model']
    elif 'state_dict' in checkpoint:
        state = checkpoint['state_dict']
    elif all(isinstance(v, torch.Tensor) for v in checkpoint.values()):
        state = checkpoint
    else:
        raise ValueError(f'Cannot identify model weights. Checkpoint keys: {list(checkpoint)}')
    state = dict(state)
    state.pop('lm_head.weight', None)
    missing, unexpected = model.load_state_dict(state, strict=False)
    assert set(missing) == {'rr_head.weight', 'rr_head.bias'} and not unexpected, (missing, unexpected)
    del state, checkpoint
    for p in model.parameters(): p.requires_grad = False
    for block in model.encoder_blocks:
        install_adapters(block.self_attn)
    
    # Decoder self-attention and cross-attention.
    for block in model.decoder_blocks:
        install_adapters(block.self_attn)
        install_adapters(block.cross_attn)

    for p in model.rr_head.parameters(): p.requires_grad = True
    model = model.to(DEVICE)
    print('Total/trainable parameters:', sum(p.numel() for p in model.parameters()),
          sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model

(…)eckpoints/encoder_decoder_mixed/model.pt:   0%|          | 0.00/318M [00:00<?, ?B/s]

In [8]:
def experiment_metadata(domain):
    return dict(domain=domain, phase1_tag=PHASE1_TAG, model_path=MODEL_PATH, tokenizer_path=TOKENIZER_PATH,
                labels=LABELS, model_config=MODEL_CONFIG, adapter_config=ADAPTER_CONFIG,
                max_seq_len=MAX_SEQ_LEN, pooling='last_real_token', context='sentence_only',
                technique='LoRMA_identity_init', head_dropout=HEAD_DROPOUT,
                seed=SEED, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                patience=PATIENCE, min_delta=MIN_DELTA)

def save_rr_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), f'{BUCKET_ROOT}/checkpoints/{tag}/checkpoint.pt')

def load_rr_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        fs.get_file(f'{BUCKET_ROOT}/checkpoints/{tag}/checkpoint.pt', str(path))
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    assert payload['metadata'] == experiment_metadata(domain), 'Checkpoint configuration mismatch'
    model.load_state_dict(payload['model'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload

@torch.no_grad()
def evaluate_rr(model, loader):
    model.eval()
    targets, predictions = [], []
    total_loss = total_n = 0
    for batch in loader:
        y = batch['labels'].to(DEVICE)
        logits = model(batch['input_ids'].to(DEVICE), batch['lengths'].to(DEVICE))
        total_loss += F.cross_entropy(logits, y, reduction='sum').item()
        total_n += len(y)
        targets.extend(y.cpu().tolist())
        predictions.extend(logits.argmax(-1).cpu().tolist())
    assert total_n > 0, 'Development set has no sentences'
    p, r, f, support = precision_recall_fscore_support(targets, predictions, labels=list(range(13)), zero_division=0)
    return dict(loss=total_loss / total_n, macro_f1=float(f.mean()), macro_precision=float(p.mean()),
                macro_recall=float(r.mean()), per_label={label: dict(precision=float(p[i]), recall=float(r[i]),
                f1=float(f[i]), support=int(support[i])) for i, label in enumerate(LABELS)},
                confusion_matrix=confusion_matrix(targets, predictions, labels=list(range(13))).tolist())

def evaluate_joint(model):
    scores = {d: evaluate_rr(model, make_loader(d, 'dev')) for d in ['CL', 'IT']}
    scores['pooled'] = evaluate_rr(model, make_loader('JOINT', 'dev'))
    scores['selection_macro_f1'] = (scores['CL']['macro_f1'] + scores['IT']['macro_f1']) / 2
    return scores


In [9]:
def train_rr(domain='JOINT'):
    assert domain == 'JOINT'
    seed_everything(SEED)
    model = build_rr_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    tag = f'rr_{PHASE1_TAG}_lorma'
    best_tag = tag + '_best'
    start = 0
    extra = dict(best_f1=-1.0, patience_reference=-1.0, stale=0, history=[], best_tag=best_tag)
    if RESUME_TAGS[domain]:
        checkpoint = load_rr_checkpoint(RESUME_TAGS[domain], model, domain, optimizer, restore_rng=True)
        start, extra = checkpoint['epoch'], checkpoint['extra']
        best_tag = extra['best_tag']
        del checkpoint
    train_loader, dev_loader = make_loader(domain, 'train'), make_loader(domain, 'dev')
    assert len(train_loader) > 0
    for epoch in range(start, NUM_EPOCHS):
        if extra['stale'] >= PATIENCE: break
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_sum = train_n = group_n = 0
        for step, batch in enumerate(tqdm(train_loader, desc=f'{domain} epoch {epoch+1}')):
            y = batch['labels'].to(DEVICE)
            logits = model(batch['input_ids'].to(DEVICE), batch['lengths'].to(DEVICE))
            loss_sum = F.cross_entropy(logits, y, reduction='sum')
            loss_sum.backward()
            group_n += len(y)
            train_n += len(y)
            train_sum += loss_sum.item()
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader):
                for p in params:
                    if p.grad is not None: p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                group_n = 0
        metrics = evaluate_joint(model)
        f1 = metrics['selection_macro_f1']
        improved = f1 > extra['best_f1']
        if improved: extra['best_f1'] = f1
        if f1 > extra['patience_reference'] + MIN_DELTA:
            extra['patience_reference'], extra['stale'] = f1, 0
        else: extra['stale'] += 1
        extra['history'].append(dict(epoch=epoch+1, train_loss=train_sum/train_n, **metrics))
        print(f"Joint train loss={train_sum/train_n:.4f}, selection macro-F1={f1:.4f}, CL={metrics['CL']['macro_f1']:.4f}, IT={metrics['IT']['macro_f1']:.4f}, pooled={metrics['pooled']['macro_f1']:.4f}, patience={extra['stale']}/{PATIENCE}")
        if improved: save_rr_checkpoint(best_tag, model, optimizer, epoch+1, extra, domain)
        save_rr_checkpoint(tag, model, optimizer, epoch+1, extra, domain)
    load_rr_checkpoint(best_tag, model, domain)
    metrics = evaluate_joint(model)
    with open(WORK_DIR / f'{domain}_dev_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
    with open(WORK_DIR / f'{domain}_history.json', 'w') as f: json.dump(extra['history'], f, indent=2)
    print('Best development metrics:', metrics)
    del optimizer
    return model, extra['history']


In [10]:
@torch.no_grad()
def predict_rr(model, domain):
    model.eval()
    docs = data[domain]['test']
    predictions = {d['id']: [None] * len(d['text']) for d in docs}
    seen = set()
    for batch in tqdm(make_loader(domain, 'test'), desc=f'{domain} test predictions'):
        logits = model(batch['input_ids'].to(DEVICE), batch['lengths'].to(DEVICE))
        for doc_id, index, predicted_id in zip(batch['doc_ids'], batch['indices'], logits.argmax(-1).cpu().tolist()):
            assert (doc_id, index) not in seen
            seen.add((doc_id, index))
            predictions[doc_id][index] = id2label[predicted_id]
    validate_predictions(predictions, docs)
    return predictions

def validate_predictions(predictions, documents):
    assert set(predictions) == {d['id'] for d in documents}
    for doc in documents:
        labels = predictions[doc['id']]
        assert len(labels) == len(doc['text']), doc['id']
        assert all(isinstance(l, str) and l in label2id for l in labels), doc['id']

joint_model, joint_history = train_rr()
cl_predictions = predict_rr(joint_model, 'CL')
it_predictions = predict_rr(joint_model, 'IT')
for domain, output in [('CL', cl_predictions), ('IT', it_predictions)]:
    with open(WORK_DIR / f'{domain}_prediction.json', 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
del joint_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Total/trainable parameters: 80429453 929933


JOINT epoch 1:   0%|          | 0/4212 [00:00<?, ?it/s]

Joint train loss=1.9858, selection macro-F1=0.0910, CL=0.0967, IT=0.0854, pooled=0.0914, patience=0/5


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (94 outputs left out)

New Data Upload: |          |  0.00B /  0.00B            

Best development metrics: {'CL': {'loss': 1.9497729480309267, 'macro_f1': 0.15387890012182995, 'macro_precision': 0.18415268240884314, 'macro_recall': 0.17111931112930595, 'per_label': {'Fact': {'precision': 0.29457364341085274, 'recall': 0.4578313253012048, 'f1': 0.3584905660377358, 'support': 249}, 'Issue': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 3}, 'ArgumentPetitioner': {'precision': 0.4444444444444444, 'recall': 0.11627906976744186, 'f1': 0.18433179723502305, 'support': 344}, 'ArgumentRespondent': {'precision': 0.2903225806451613, 'recall': 0.038135593220338986, 'f1': 0.06741573033707865, 'support': 236}, 'Statute': {'precision': 0.6271186440677966, 'recall': 0.6379310344827587, 'f1': 0.6324786324786325, 'support': 58}, 'Dissent': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 0}, 'PrecedentReliedUpon': {'precision': 0.25882352941176473, 'recall': 0.15492957746478872, 'f1': 0.19383259911894274, 'support': 142}, 'PrecedentNotReliedUpon': {'precision': 0.0, 

CL test predictions:   0%|          | 0/335 [00:00<?, ?it/s]

IT test predictions:   0%|          | 0/215 [00:00<?, ?it/s]

In [11]:
# Reload saved domain predictions so packaging can be rerun without retraining.
with open(WORK_DIR / 'CL_prediction.json', encoding='utf-8') as f: cl_predictions = json.load(f)
with open(WORK_DIR / 'IT_prediction.json', encoding='utf-8') as f: it_predictions = json.load(f)
assert not (cl_predictions.keys() & it_predictions.keys()), 'Cross-domain document ID collision'
predictions = {**cl_predictions, **it_predictions}
validate_predictions(predictions, data['CL']['test'] + data['IT']['test'])
json_path = WORK_DIR / 'prediction.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
zip_path = WORK_DIR / f"rr_{PHASE1_TAG}_lorma_joint.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname='prediction.json')
print('JSON:', json_path)
print('ZIP:', zip_path)
print('Documents:', len(predictions), 'sentences:', sum(map(len, predictions.values())))

JSON: /kaggle/working/rr_encoder_decoder_mixed_lorma_joint/prediction.json
ZIP: /kaggle/working/rr_encoder_decoder_mixed_lorma_joint/rr_encoder_decoder_mixed_lorma_joint.zip
Documents: 10 sentences: 2197


In [12]:
# import re
# import torch
# from pathlib import Path

# SOURCE_PATH = (
#     "foundation/checkpoints/"
#     "legal_foundation_129m_single_v3/latest_checkpoint.pt"
# )

# RENAMED_PATH = (
#     "foundation/checkpoints/"
#     "legal_foundation_129m_single_v3_rr_names/model.pt"
# )

# local_dir = Path("/kaggle/working/renamed_phase1")
# local_dir.mkdir(parents=True, exist_ok=True)

# source_file = local_dir / "latest_checkpoint.pt"
# renamed_file = local_dir / "model.pt"

# fs.get_file(
#     f"{BUCKET_ROOT}/{SOURCE_PATH}",
#     str(source_file),
# )

# checkpoint = torch.load(
#     source_file,
#     map_location="cpu",
#     weights_only=False,
# )

# for key in ["model_state_dict", "model", "state_dict"]:
#     if key in checkpoint:
#         state = checkpoint[key]
#         break
# else:
#     if not all(
#         isinstance(value, torch.Tensor)
#         for value in checkpoint.values()
#     ):
#         raise ValueError(
#             f"Cannot identify model weights: {list(checkpoint)}"
#         )
#     state = checkpoint

# component_map = {
#     "n1": "norm1",
#     "n2": "norm2",
#     "a.q": "attention.q_proj",
#     "a.k": "attention.k_proj",
#     "a.v": "attention.v_proj",
#     "a.o": "attention.out_proj",
#     "ff.g": "ffn.gate_proj",
#     "ff.u": "ffn.up_proj",
#     "ff.d": "ffn.down_proj",
# }

# renamed_state = {}

# for key, value in state.items():
#     if key == "emb.weight":
#         new_key = "token_embedding.weight"
#     elif key == "norm.weight":
#         new_key = "final_norm.weight"
#     else:
#         match = re.fullmatch(
#             r"blocks\.(\d+)\.(n1|n2|a\.[qkvo]|ff\.[gud])\.weight",
#             key,
#         )
#         if match:
#             layer, component = match.groups()
#             new_key = (
#                 f"blocks.{layer}.{component_map[component]}.weight"
#             )
#         else:
#             new_key = key

#     assert new_key not in renamed_state, new_key
#     renamed_state[new_key] = value

# assert "token_embedding.weight" in renamed_state
# assert "final_norm.weight" in renamed_state
# assert "blocks.0.attention.q_proj.weight" in renamed_state

# # Save model weights only, without optimizer/training state.
# torch.save(renamed_state, renamed_file)

# # Verify the saved tensors before uploading.
# saved_state = torch.load(
#     renamed_file,
#     map_location="cpu",
#     weights_only=True,
# )
# assert saved_state.keys() == renamed_state.keys()
# for key in renamed_state:
#     torch.testing.assert_close(
#         saved_state[key],
#         renamed_state[key],
#         rtol=0,
#         atol=0,
#         equal_nan=True,
#     )

# fs.put_file(
#     str(renamed_file),
#     f"{BUCKET_ROOT}/{RENAMED_PATH}",
# )

# print("Uploaded:", RENAMED_PATH)